In [ ]:
import pandas as pd
import numpy as np
import re
path = "enriched_collision_data_with_weather.csv"  # change if needed

# Tune these if needed
WIND_HIGH_THRESHOLD = 18.0
PRECIP_NONZERO_THRESHOLD = 0.01
SHOW_MISMATCH_LIMIT = 100

# Inferred mapping from your dataset patterns.
# Replace this with the official legend if you have it.
CODE_TO_LABEL = {
    1: "Fine no high winds",
    2: "Raining no high winds",
    3: "Snowing no high winds",
    4: "Fine + high winds",
    5: "Raining + high winds",
    6: "Snowing + high winds",
    7: "Fog or mist",
    8: "Other",
    9: "Unknown",
}

NUMERIC_COLS = ["precipitation", "rain", "showers", "snowfall", "windspeed_10m"]

df = pd.read_csv(path)

# Coerce numeric columns
for c in NUMERIC_COLS + ["weather_conditions"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

if "weather_conditions" not in df.columns:
    raise ValueError("Missing weather_conditions column")

missing = [c for c in NUMERIC_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Human-readable labels
df["weather_label"] = df["weather_conditions"].map(CODE_TO_LABEL).fillna(
    "UnknownCode_" + df["weather_conditions"].astype("Int64").astype(str)
)

# Actual signals from numeric weather fields
df["any_precip"] = (
    (df["precipitation"] > PRECIP_NONZERO_THRESHOLD)
    | (df["rain"] > PRECIP_NONZERO_THRESHOLD)
    | (df["showers"] > PRECIP_NONZERO_THRESHOLD)
    | (df["snowfall"] > PRECIP_NONZERO_THRESHOLD)
)
df["any_rain"] = (df["rain"] > PRECIP_NONZERO_THRESHOLD) | (df["precipitation"] > PRECIP_NONZERO_THRESHOLD)
df["any_snow"] = df["snowfall"] > PRECIP_NONZERO_THRESHOLD
df["high_wind"] = df["windspeed_10m"] >= WIND_HIGH_THRESHOLD

def infer_expected_flags(label):
    s = str(label).lower()
    return {
        "expects_fine": ("fine" in s) or ("clear" in s),
        "expects_rain": ("rain" in s) or ("showers" in s),
        "expects_snow": ("snow" in s),
        "expects_high_wind": bool(re.search(r'(?<!no )high winds?', s)),
        "expects_low_wind": "no high winds" in s,
        "expects_fog": ("fog" in s) or ("mist" in s),
    }

flags = df["weather_label"].apply(infer_expected_flags).apply(pd.Series)
df = pd.concat([df, flags], axis=1)

# Build mismatch reasons
df["mismatch_reason"] = ""

# Fine/clear should usually not have precipitation or snow
mask = df["expects_fine"] & (df["any_precip"] | df["any_snow"])
df.loc[mask, "mismatch_reason"] += "fine_but_precip_or_snow; "

# Rain labels should usually have rain/precip
mask = df["expects_rain"] & ~df["any_precip"]
df.loc[mask, "mismatch_reason"] += "rain_label_but_no_precip; "

# Snow labels should usually have snowfall
mask = df["expects_snow"] & ~df["any_snow"]
df.loc[mask, "mismatch_reason"] += "snow_label_but_no_snowfall; "

# Fog/mist usually low precip, but keep this weak so you can inspect only obvious issues
mask = df["expects_fog"] & (df["any_snow"] | (df["rain"] > 0.5))
df.loc[mask, "mismatch_reason"] += "fog_label_but_snow_or_heavy_rain; "

# Wind checks
mask = df["expects_high_wind"] & ~df["high_wind"]
df.loc[mask, "mismatch_reason"] += "highwind_label_but_low_wind; "

mask = df["expects_low_wind"] & df["high_wind"]
df.loc[mask, "mismatch_reason"] += "lowwind_label_but_high_wind; "

df["mismatch_reason"] = df["mismatch_reason"].str.replace(r";\s*$", "", regex=True).fillna("")

# Summary
summary = (
    df.assign(has_mismatch=df["mismatch_reason"] != "")
      .groupby(["weather_conditions", "weather_label"], dropna=False)
      .agg(
          rows=("weather_conditions", "size"),
          mismatches=("has_mismatch", "sum"),
          mismatch_rate=("has_mismatch", "mean"),
          median_wind=("windspeed_10m", "median"),
          cols_with_precip_non_zero=("any_precip", "mean"),
          cols_with_rain_nonzero=("any_rain", "mean"),
          cols_with_snow_nonzero=("any_snow", "mean"),
      )
      .sort_values(["mismatch_rate", "rows"], ascending=[False, False])
)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)

print("\n=== Summary ===")
print(summary.to_string(float_format=lambda x: f"{x:0.3f}"))

# Show first 100 mismatches side by side
mismatches = df[df["mismatch_reason"] != ""].copy()

print(f"\n=== First {SHOW_MISMATCH_LIMIT} mismatches side by side ===")
if mismatches.empty:
    print("No mismatches found.")
else:
    cols = [
        "weather_conditions", "weather_label",
        "precipitation", "rain", "showers", "snowfall", "windspeed_10m",
        "any_precip", "any_rain", "any_snow", "high_wind",
        "mismatch_reason",
    ]
    print(mismatches[cols].head(SHOW_MISMATCH_LIMIT).to_string(index=False))

# Optional: mismatch reasons breakdown
print("\n=== Mismatch reason counts ===")
reason_counts = (
    mismatches["mismatch_reason"]
    .str.split("; ")
    .explode()
    .value_counts()
)
print(reason_counts.to_string())